# (u, v) coordinates and an evenly spaced mesh on the PLS surface

For each model this notebook reads the surface that `notebooks/3_pls_arc_length_surface.ipynb` saved in `artifacts/<model>/pls/shape/`. It builds coordinates (u, v) on it that follow the gradient of `bcpc_arc_length`, lays a mesh evenly spaced in those coordinates, and saves the readout that maps new points to (u, v). It does not read the stakes classes. The BCPC's `cv_fold` column is dropped too, because its folds were dealt by class. `bcpc_arc_length` only sets the direction of the coordinates and colours the mesh; the readout is the pair of surface lengths (u, v).

1. **Read the surface.** The PLS transform, the convex B-spline surface and every row's closest point on it. The files' checksums and their link to the BCPC bundle are checked, and each saved closest point must reproduce its saved distance.
2. **Coordinates and mesh.** A smooth field is fitted on the surface to the rows' `bcpc_arc_length` at their closest points. It only gives the direction in which the arc length increases. The *start curve* is the field's level curve through the rows' weighted median `bcpc_arc_length`, and *gradient lines* run along the field's surface gradient across it.
   - **u**: signed surface length along a point's gradient line from the start curve, positive towards larger `bcpc_arc_length`.
   - **v**: surface length along the start curve to where the point's gradient line crosses it, measured from where the curve passes closest to the rows' weighted mean. v is constant along each gradient line.

   The mesh nodes sit at equal steps of u along gradient lines that are equally spaced in v, so they are evenly spaced in surface length along both families of lines. The sides along the gradient lines follow the gradient exactly. The sides of constant u are not level curves of the field wherever the arc length rises at different rates along a level curve, so there the cells are not rectangles.
3. **Colouring.** `bcpc_arc_length` at every mesh node is learnt from the rows. Each row is interpolated bilinearly in (u, v) from its cell's four nodes, with a smoothness penalty chosen by cross-validation over held-out tasks. This only colours the mesh; it is not the readout.
4. **Plot.** The mesh in PLS1-3, its nodes coloured by the learnt `bcpc_arc_length`.
5. **Save and read out.** The readout (PLS transform, surface, coordinate tables) and the mesh go to `artifacts/<model>/pls/mesh/`. `sm.load_mesh(config)` returns a readout that maps activations to (u, v).

Every template file gets equal total weight, split equally among its rows (the saved `pls_fit_weight`).

In [ ]:
from pathlib import Path
import sys
import time

import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts' / 'surface_mesh.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import surface_mesh as sm
from scripts.pipeline_config import discover_run_dirs

pd.set_option('display.width', 160)
print('Repository:', ROOT)

## Configuration

`MODELS = None` takes every model directory under `ARTIFACT_ROOT` with a saved surface (`pls/shape/model.json`). A list of directory names restricts the run to those models.

- `LEVELS`: how many nodes each gradient line has over the rows' range of u. It sets the u step, the surface length between neighbouring nodes along a gradient line.
- `LINES`: how many gradient lines span the rows' range of v. It sets the v step, the surface length between neighbouring lines along the start curve.
- `SMOOTHING`: candidate weights of the colouring's smoothness penalty. The loss is the weighted mean squared error plus the weight times the mean squared second difference of the node values. `FOLDS` held-out-task folds choose among them by the one-standard-error rule, which takes the smoothest candidate whose held-out error is within one standard error of the lowest.
- `PLOT_ROWS`: how many rows the plot carries, hidden until *Rows* is clicked in the legend. 0 leaves them out.
- `SAVE`: write each model's readout and mesh to `pls/mesh/`.

Runtime is about 35 s per model: 25 s to build the coordinate tables, 3 s to read out every row and 6 s for the colouring's cross-validation.

In [ ]:
ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'
MODELS = None                       # None = every model with a saved surface

LEVELS = 41                         # nodes along each gradient line (sets the u step)
LINES = 51                          # gradient lines (sets the v step)
SMOOTHING = sm.MeshSettings().smoothing   # candidate penalty weights of the colouring
FOLDS = 5
PLOT_ROWS = 4_000                   # rows in the plot, hidden until toggled; 0 = none
SAVE = True                         # write the readout and mesh to pls/mesh/

SETTINGS = sm.MeshSettings(levels=LEVELS, lines=LINES, smoothing=SMOOTHING, folds=FOLDS)

## Models

In [ ]:
def has_shape(run_dir):
    return (run_dir / 'pls' / 'shape' / 'model.json').is_file()


run_dirs = discover_run_dirs(ARTIFACT_ROOT)
if MODELS is not None:
    missing = [name for name in MODELS if ARTIFACT_ROOT / name not in run_dirs or not has_shape(ARTIFACT_ROOT / name)]
    if missing:
        raise ValueError(f'No saved surface for {missing} under {ARTIFACT_ROOT}.')
    run_dirs = [ARTIFACT_ROOT / name for name in MODELS]
skipped = [path.name for path in run_dirs if not has_shape(path)]
run_dirs = [path for path in run_dirs if has_shape(path)]
if skipped:
    print('No saved surface, skipped:', ', '.join(skipped))
if not run_dirs:
    raise ValueError(f'No model under {ARTIFACT_ROOT} has a saved surface; '
                     'run notebooks/3_pls_arc_length_surface.ipynb first.')
print(f'{len(run_dirs)} model(s):', ', '.join(path.name for path in run_dirs))

## 1. Read the surface artifacts

`sm.load_surface` reads `pls/shape/model.json`, `model.npz` and `rows.parquet` through `bs.load_shape`. The summary shows a few of the surface's fit diagnostics from notebook 3.

In [ ]:
surfaces = {}
for run_dir in run_dirs:
    data = sm.load_surface(run_dir)
    surfaces[run_dir.name] = data
    fit, arc = data.metadata['shape_fit'], data.rows[sm.TARGET]
    print(f'=== {run_dir.name} ===  {len(data.rows):,} rows, surface fitted in PLS1-{len(data.pls_columns)} '
          f'on {data.metadata["created_utc"]}')
    print(f'  weighted RMS distance to the surface {fit["weighted_rms_distance"]:.4g}; '
          f'variance share on the surface {fit["variance_share_on_surface"]:.4f}; '
          f'{sm.TARGET} from {arc.min():.4g} to {arc.max():.4g}')

## 2. Build the coordinates and the mesh

`sm.build_mesh` fits the arc-length field, builds the coordinate tables and traces the mesh.

- `field_weighted_r2` and `field_weighted_rms_error` show how well the smooth field follows `bcpc_arc_length`. The field is used only for its direction.
- `u_step` and `v_step` are the mesh's spacing in surface length (PLS units). `rows_u_*` and `rows_v_*` give the rows' range of u and v.
- `node_readout_max_abs_u` and `node_readout_max_abs_v`: the mesh nodes, traced by integrating along the gradient lines, read back through the coordinate tables must return their own u and v. The `round_trip_*` rows check the tables against each other the same way.
- `nodes_on_data` counts the nodes inside the convex hull of the rows' closest points. Only those are drawn.

In [ ]:
meshes = {}
for name, data in surfaces.items():
    started = time.perf_counter()
    meshes[name] = sm.build_mesh(data, SETTINGS)
    print(f'=== {name} ===  {time.perf_counter() - started:.1f} s')
    print(meshes[name].diagnostics.to_string(float_format=lambda x: f'{x:.4g}'))

## 3. Colour the mesh by bcpc_arc_length

Every row is first read out exactly as a new point would be: its closest point on the surface is found anew from its PLS scores (`closest_point_moved` marks the few rows that lie about equally close to two parts of the surface and settle on the other one), then its (u, v) fixes its mesh cell. `sm.fit_mapping` learns a `bcpc_arc_length` value at every node. The cross-validation folds hold out whole tasks, every template of a task together, dealt stratified by task mean `bcpc_arc_length`.

- `cv_rms_error` and `cv_q2` are the held-out-task error and R2 at the chosen penalty. `weighted_r2` is the in-sample fit.
- `variance_share_at_equal_u`: the share of the learnt node values' variance that lies across the gradient lines at equal u. It would be 0 if every row of nodes at equal u were a level curve of `bcpc_arc_length`.
- `cell_side_cos_median` and `cell_side_cos_p95`: |cos| between each cell's two sides at a corner, as straight chords between nodes in all PLS components. The sides along the gradient lines follow the gradient; this measures how far the sides of constant u tilt away from orthogonal to them.
- `readout_status` counts rows whose closest point lies on the patch edge (`patch_boundary`) or outside the convex hull of the rows' closest points (`outside_data`).

In [ ]:
mappings = {}
for name, data in surfaces.items():
    mapping = sm.fit_mapping(data, meshes[name], SETTINGS)
    mappings[name] = mapping
    a = mapping.selection.attrs
    print(f'=== {name} ===  penalty weight {a["one_se"]:.3g} (one-SE rule; lowest held-out error at {a["best"]:.3g})')
    if a['at_grid_edge']:
        print('  the chosen weight is the largest in SMOOTHING; extend the grid to see where the error turns.')
    print(mapping.diagnostics.to_string(float_format=lambda x: f'{x:.4g}'))
    print(mapping.rows['readout_status'].value_counts().to_string())
    print(mapping.selection.assign(cv_rms=mapping.selection['cv_mse'] ** 0.5).to_string(float_format=lambda x: f'{x:.4g}'))

## 4. The mesh in PLS1-3

The mesh lives in every saved PLS component. The plot shows its projection onto PLS1-3, so its spacing and angles look distorted wherever the surface also bends through the other components. Dark lines are the gradient lines (constant v) and grey lines the lines of constant u. Nodes are coloured by the learnt `bcpc_arc_length`, and hovering over a node shows its u, v, value and `refined_stakes` equivalent. Only nodes inside the convex hull of the rows' closest points and in cells that hold rows are drawn. Click *Rows* in the legend to show the rows, coloured on the same scale.

In [ ]:
for name, data in surfaces.items():
    sm.plot_mesh(data, meshes[name], mappings[name], max_rows=PLOT_ROWS).show()

## 5. Save the readout and read points out

`sm.save_mesh` writes each model's readout and mesh to `artifacts/<model>/pls/mesh/`:

- `model.npz`: the PLS transform, the surface, the arc-length field, the coordinate tables and the mesh (`mesh_u`, `mesh_v`, `mesh_st`, `mesh_arc_length`)
- `model.json`: what u, v and `readout_status` mean, the settings, the diagnostics above, and checksums of the files, the code and the saved surface it was built on
- `rows.parquet`: every row with its readout (`u`, `v`, closest point, `surface_distance`, `readout_status`) and its mesh cell; no stakes classes

Each saved readout is reloaded and must reproduce the saved rows' (u, v), both at their closest points and read out again from their PLS scores.

`readout, metadata, arrays = sm.load_mesh(config)` loads it again. `readout(X)` maps raw activations (n, features) to a table with `u`, `v`, `surface_s`, `surface_t`, `surface_distance` and `readout_status`, and `readout.from_scores(scores)` does the same from PLS scores. The cell below reads out a few saved rows from their PLS scores as an example.

In [ ]:
if SAVE:
    for name, data in surfaces.items():
        directory = sm.save_mesh(data, meshes[name], mappings[name], notebook='notebooks/4_surface_mesh.ipynb')
        readout, metadata, _ = sm.load_mesh(data.config)
        sample = data.rows.sample(5, random_state=0)
        example = readout.from_scores(sample[data.pls_columns].to_numpy()).set_index(sample.index)
        print(f'{name}: saved to {directory}')
        print(sample[['task', sm.TARGET]].join(example[['u', 'v', 'surface_distance', 'readout_status']])
              .to_string(float_format=lambda x: f'{x:.4g}'))